# Stage 2: Tooth Anomaly Classifier

EfficientNet-B0 (or ResNet-18) binary classifier trained on DENTEX Challenge 2023 data.

**Prerequisite:** Run `stage1_segmentation/teeth_segmentation.ipynb` first in the same Kaggle session so that `model` and `test_dataset` are in memory.

**Pipeline role:** Takes each tooth crop from Stage 1 bounding boxes and classifies it as **Normal** or **Anomaly** (Caries / Periapical Lesion / Deep Caries / Impacted Tooth).

**Improvements over v1:**
- EfficientNet-B0 backbone (lighter, more accurate than ResNet-18; configurable)
- Label smoothing in BCE loss (reduces overconfidence)
- AUC-ROC tracked at every epoch and on test set
- Early stopping (no improvement in val F1 for N epochs → stop)
- Batch-mode bridge inference helper for Stage 1 → Stage 2
- Confusion matrix heatmap in test evaluation

In [ ]:
# ================== CELL 1: IMPORTS ==================
print('='*60)
print('STAGE 2 — CELL 1: Importing libraries')
print('='*60)

import os, json, time, random, datetime
from collections import defaultdict
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
try:
    from sklearn.metrics import roc_auc_score
    SKLEARN_OK = True
except ImportError:
    SKLEARN_OK = False
    print('  ⚠ scikit-learn not found — AUC-ROC will use manual trapezoid approximation')

print(f'  ✓ PyTorch      : {torch.__version__}')
print(f'  ✓ CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  ✓ GPU          : {torch.cuda.get_device_name(0)}')
    print(f'  ✓ VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('  ✓ All imports complete')

In [ ]:
# ================== CELL 2: CONFIG ==================
print('='*60)
print('STAGE 2 — CELL 2: Configuration')
print('='*60)

CONFIG_S2 = {
    'seed'              : 42,
    'batch_size'        : 32,
    'lr'                : 1e-4,
    'epochs'            : 20,
    'img_size'          : 224,
    'num_workers'       : 2,
    'binary_threshold'  : 0.5,
    'crop_padding'      : 8,
    'weight_decay'      : 1e-4,
    'lr_patience'       : 3,
    'lr_factor'         : 0.5,
    'early_stop_patience': 6,
    'label_smoothing'   : 0.05,
    'backbone'          : 'efficientnet_b0',
    'freeze_backbone'   : False,
}

# ── Step 1: Find the DENTEX base directory ─────────────────────────────
_CANDIDATE_BASES = [
    '/kaggle/input/dentex-challenge-2023',
    '/kaggle/input/dentex',
    '/kaggle/input/dentex-2023',
]
DENTEX_BASE = next((p for p in _CANDIDATE_BASES if os.path.isdir(p)), None)

if DENTEX_BASE is None:
    # Last resort: scan all of /kaggle/input/ for any dir containing .json or .png files
    print('  ⚠ Known DENTEX paths not found. Scanning /kaggle/input/ ...')
    for d in sorted(os.listdir('/kaggle/input')):
        full = os.path.join('/kaggle/input', d)
        if os.path.isdir(full):
            contents = os.listdir(full)
            print(f'    Found: {full}  ({len(contents)} items)')
    raise RuntimeError(
        'DENTEX dataset not found under /kaggle/input/. '
        'Please add truthisneverlinear/dentex-challenge-2023 via '
        'Notebook Settings → Add Data.'
    )

print(f'  ✓ DENTEX base   : {DENTEX_BASE}')

# ── Step 2: Walk the dataset tree to find images dir + annotation file ─
print('  Scanning dataset structure...')

def _walk_find(base, match_fn, max_depth=4):
    """Walk up to max_depth levels; return first path where match_fn(path) is True."""
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, '').count(os.sep)
        if depth > max_depth:
            dirs[:] = []  # prune
            continue
        for name in files + dirs:
            full = os.path.join(root, name)
            if match_fn(full, name):
                return full
    return None

# Find image directory: directory that contains at least one .jpg or .png file
def _is_image_dir(path, name):
    if not os.path.isdir(path):
        return False
    files = os.listdir(path)
    return any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files)

# Priority: prefer dirs named 'train', 'images', 'xrays'
_PREFERRED_IMG_DIRS = ['train', 'images', 'xrays']
DENTEX_IMG_DIR = None
for pref in _PREFERRED_IMG_DIRS:
    candidate = os.path.join(DENTEX_BASE, pref)
    if os.path.isdir(candidate) and any(
        f.lower().endswith(('.jpg','.jpeg','.png'))
        for f in os.listdir(candidate)
    ):
        DENTEX_IMG_DIR = candidate
        break

if DENTEX_IMG_DIR is None:
    # Walk to find any image-containing directory
    DENTEX_IMG_DIR = _walk_find(DENTEX_BASE, _is_image_dir)

if DENTEX_IMG_DIR is None:
    DENTEX_IMG_DIR = DENTEX_BASE  # last resort: root itself

# Find annotation JSON: prefer files with 'disease' or 'enumeration' in name
def _is_json_file(path, name):
    return name.endswith('.json')

_ANN_PRIORITY = [
    'train_quadrant_enumeration_disease.json',
    'train_quadrant_enumeration_disease_100.json',
    'annotations.json',
    'train.json',
    'annotation.json',
]
DENTEX_ANN_PATH = None
for ann_name in _ANN_PRIORITY:
    # Search both at root and one level deep
    for check in [
        os.path.join(DENTEX_BASE, ann_name),
        *[os.path.join(DENTEX_BASE, sub, ann_name)
          for sub in os.listdir(DENTEX_BASE)
          if os.path.isdir(os.path.join(DENTEX_BASE, sub))]
    ]:
        if os.path.exists(check):
            DENTEX_ANN_PATH = check
            break
    if DENTEX_ANN_PATH:
        break

if DENTEX_ANN_PATH is None:
    # Walk to find any JSON that looks like a COCO annotation (has 'annotations' key)
    print('  Priority annotation files not found — scanning for COCO JSON...')
    for root, _, files in os.walk(DENTEX_BASE):
        for fname in files:
            if not fname.endswith('.json'):
                continue
            fpath = os.path.join(root, fname)
            try:
                with open(fpath) as _f:
                    _d = json.load(_f)
                if isinstance(_d, dict) and ('annotations' in _d or 'images' in _d):
                    DENTEX_ANN_PATH = fpath
                    break
                elif isinstance(_d, list) and len(_d) > 0:
                    DENTEX_ANN_PATH = fpath
                    break
            except Exception:
                continue
        if DENTEX_ANN_PATH:
            break

if DENTEX_ANN_PATH is None:
    raise RuntimeError(
        f'No annotation JSON found under {DENTEX_BASE}. '
        'Please check the dataset structure.'
    )

# ── Step 3: Print full discovered layout ───────────────────────────────
print(f'\n  Dataset layout discovered:')
print(f'    Base dir         : {DENTEX_BASE}')
for item in sorted(os.listdir(DENTEX_BASE)):
    ipath = os.path.join(DENTEX_BASE, item)
    if os.path.isdir(ipath):
        n = len(os.listdir(ipath))
        print(f'      📁 {item}/  ({n} items)')
    else:
        size_kb = os.path.getsize(ipath) // 1024
        print(f'      📄 {item}  ({size_kb} KB)')

n_imgs = len([f for f in os.listdir(DENTEX_IMG_DIR)
              if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f'\n  ✓ Image dir        : {DENTEX_IMG_DIR}  ({n_imgs} images)')
print(f'  ✓ Annotation file  : {DENTEX_ANN_PATH}  ({os.path.getsize(DENTEX_ANN_PATH)//1024} KB)')

CHECKPOINT_BEST  = '/kaggle/working/stage2_anomaly_best.pth'
CHECKPOINT_FINAL = '/kaggle/working/stage2_anomaly_final.pth'
REPORT_PATH      = '/kaggle/working/stage2_report.json'

random.seed(CONFIG_S2['seed'])
np.random.seed(CONFIG_S2['seed'])
torch.manual_seed(CONFIG_S2['seed'])
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CONFIG_S2['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('\n  Stage 2 configuration:')
for k, v in CONFIG_S2.items(): print(f'    {k:<26}: {v}')
print(f'  Device             : {device}')
print('  ✓ Config complete')

In [ ]:
# ================== CELL 3: LOAD DENTEX ANNOTATIONS ==================
print('='*60)
print('STAGE 2 — CELL 3: Loading DENTEX annotations')
print('='*60)

DISEASE_NAMES = {'caries', 'deep caries', 'periapical lesion', 'impacted tooth'}

def _label_from_raw(diagnosis):
    if diagnosis is None: return 0
    if isinstance(diagnosis, (int, float)): return 1 if diagnosis > 0 else 0
    if isinstance(diagnosis, str): return 1 if diagnosis.strip().lower() in DISEASE_NAMES else 0
    if isinstance(diagnosis, list): return 1 if len(diagnosis) > 0 else 0
    return 0

def load_dentex_annotations(annotation_path):
    t0 = time.time()
    print(f'  Loading: {annotation_path}')
    with open(annotation_path) as f:
        data = json.load(f)
    samples, skipped_bbox, skipped_file = [], 0, 0

    if isinstance(data, dict) and 'annotations' in data and 'images' in data:
        print('  Detected format  : COCO-style JSON')
        id_to_file = {img['id']: img['file_name'] for img in data['images']}
        cat_id_to_name = {cat['id']: cat['name'].strip().lower()
                          for cat in data.get('categories', [])}
        print(f'  Categories       : {list(cat_id_to_name.values())}')
        disease_cat_ids = {cid for cid, name in cat_id_to_name.items()
                           if any(d in name for d in DISEASE_NAMES)}
        print(f'  Disease cat IDs  : {disease_cat_ids}')
        anns_by_image = defaultdict(list)
        for ann in data['annotations']:
            anns_by_image[ann['image_id']].append(ann)
        for image_id, anns in anns_by_image.items():
            fname = id_to_file.get(image_id)
            if not fname: skipped_file += 1; continue
            for ann in anns:
                bbox = ann.get('bbox')
                if not bbox or len(bbox) != 4 or bbox[2] <= 1 or bbox[3] <= 1:
                    skipped_bbox += 1; continue
                x, y, w, h = bbox
                cat_id = ann.get('category_id', -1)
                label = (1 if (disease_cat_ids and cat_id in disease_cat_ids)
                         else _label_from_raw(ann.get('diagnosis', ann.get('label'))))
                samples.append({'file_name': fname,
                                'bbox': [float(x), float(y), float(x+w), float(y+h)],
                                'label': label})
    elif isinstance(data, list):
        print('  Detected format  : List-style JSON')
        for item in data:
            fname = item.get('file_name') or item.get('image')
            for tooth in item.get('teeth', []):
                bbox = tooth.get('bbox')
                if not bbox or len(bbox) != 4: skipped_bbox += 1; continue
                x1,y1,x2,y2 = bbox
                if (x2-x1) <= 1 or (y2-y1) <= 1: skipped_bbox += 1; continue
                samples.append({'file_name': fname,
                                'bbox': [float(x1),float(y1),float(x2),float(y2)],
                                'label': _label_from_raw(tooth.get('diagnosis', tooth.get('label')))})
    else:
        raise ValueError(f'Unsupported annotation format: {type(data)}')

    n_total  = len(samples)
    n_anomaly = sum(s['label'] for s in samples)
    n_normal  = n_total - n_anomaly
    print(f'\n  Total samples    : {n_total}')
    print(f'  Normal  (label=0): {n_normal}  ({100*n_normal/max(1,n_total):.1f}%)')
    print(f'  Anomaly (label=1): {n_anomaly} ({100*n_anomaly/max(1,n_total):.1f}%)')
    print(f'  Imbalance ratio  : {n_normal/max(1,n_anomaly):.2f}:1')
    print(f'  Skipped (bad bbox): {skipped_bbox} | Skipped (no file): {skipped_file}')
    print(f'  Parsed in        : {time.time()-t0:.2f}s')
    print('  ✓ Annotations loaded')
    return samples

all_samples = load_dentex_annotations(DENTEX_ANN_PATH)

In [ ]:
# ================== CELL 4: TOOTH CROP DATASET ==================
print('='*60)
print('STAGE 2 — CELL 4: Building ToothCropDataset')
print('='*60)

class ToothCropDataset(Dataset):
    """
    Crops individual tooth regions from full X-ray images using
    Stage 1 bounding boxes (or DENTEX ground-truth boxes).
    Applies label smoothing to targets during __getitem__ when
    smoothing > 0 (training only — pass smoothing=0 for eval).
    """
    def __init__(self, img_dir, samples, transform=None,
                 padding=8, smoothing=0.0):
        self.img_dir   = img_dir
        self.samples   = samples
        self.transform = transform
        self.padding   = padding
        self.smoothing = smoothing
        n_anom = sum(s['label'] for s in samples)
        print(f'    Dataset size : {len(samples)} '
              f'| Normal: {len(samples)-n_anom} '
              f'| Anomaly: {n_anom} '
              f'| smoothing={smoothing}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        try:
            image = Image.open(
                os.path.join(self.img_dir, s['file_name'])
            ).convert('RGB')
        except Exception:
            image = Image.new('RGB', (64, 64), (128, 128, 128))

        W, H = image.size
        x1 = max(0, int(s['bbox'][0]) - self.padding)
        y1 = max(0, int(s['bbox'][1]) - self.padding)
        x2 = min(W, int(s['bbox'][2]) + self.padding)
        y2 = min(H, int(s['bbox'][3]) + self.padding)
        crop = image.crop((x1, y1, x2, y2))

        raw_label = float(s['label'])
        if self.smoothing > 0:
            raw_label = raw_label * (1 - self.smoothing) + (1 - raw_label) * self.smoothing
        label = torch.tensor(raw_label, dtype=torch.float32)

        if self.transform:
            crop = self.transform(crop)
        return crop, label

print('  ToothCropDataset class defined (with label smoothing support)')
print('  ✓ Dataset class ready')

In [ ]:
# ================== CELL 5: SPLIT + TRANSFORMS ==================
print('='*60)
print('STAGE 2 — CELL 5: Train / Val / Test split + Transforms')
print('='*60)

random.shuffle(all_samples)
n       = len(all_samples)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)
train_s = all_samples[:n_train]
val_s   = all_samples[n_train:n_train + n_val]
test_s  = all_samples[n_train + n_val:]

print(f'  Total: {n} | Train: {n_train} ({100*n_train/n:.1f}%) '
      f'| Val: {n_val} ({100*n_val/n:.1f}%) '
      f'| Test: {len(test_s)} ({100*len(test_s)/n:.1f}%)')

train_tfms = transforms.Compose([
    transforms.Resize((CONFIG_S2['img_size'], CONFIG_S2['img_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.06, 0.06)),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tfms = transforms.Compose([
    transforms.Resize((CONFIG_S2['img_size'], CONFIG_S2['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_labels  = [s['label'] for s in train_s]
class_weights = [1.0 / max(1, train_labels.count(0)),
                 1.0 / max(1, train_labels.count(1))]
sampler = WeightedRandomSampler(
    [class_weights[l] for l in train_labels],
    len(train_labels), replacement=True
)

train_ds = ToothCropDataset(DENTEX_IMG_DIR, train_s, train_tfms,
                             CONFIG_S2['crop_padding'],
                             smoothing=CONFIG_S2['label_smoothing'])
val_ds   = ToothCropDataset(DENTEX_IMG_DIR, val_s,   eval_tfms,
                             CONFIG_S2['crop_padding'], smoothing=0.0)
test_ds  = ToothCropDataset(DENTEX_IMG_DIR, test_s,  eval_tfms,
                             CONFIG_S2['crop_padding'], smoothing=0.0)

train_loader = DataLoader(train_ds, CONFIG_S2['batch_size'], sampler=sampler,
                          num_workers=CONFIG_S2['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   CONFIG_S2['batch_size'], shuffle=False,
                          num_workers=CONFIG_S2['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  CONFIG_S2['batch_size'], shuffle=False,
                          num_workers=CONFIG_S2['num_workers'])

print(f'  Class weights → Normal: {class_weights[0]:.6f} | Anomaly: {class_weights[1]:.6f}')
print(f'  Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')
print('  ✓ DataLoaders ready')

In [ ]:
# ================== CELL 6: MODEL ==================
print('='*60)
print('STAGE 2 — CELL 6: Building anomaly classifier')
print(f'  Backbone: {CONFIG_S2["backbone"]}')
print('='*60)

def get_anomaly_classifier(backbone='efficientnet_b0', freeze_backbone=False):
    backbone = backbone.lower()
    if backbone == 'efficientnet_b0':
        net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        if freeze_backbone:
            for name, param in net.named_parameters():
                if 'features.7' not in name and 'features.8' not in name and 'classifier' not in name:
                    param.requires_grad = False
            print('  Backbone frozen  : all except features.7, features.8, classifier')
        else:
            print('  Backbone         : fully trainable (EfficientNet-B0)')
        in_features = net.classifier[1].in_features
        net.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, 1)
        )
    elif backbone == 'resnet18':
        net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        if freeze_backbone:
            for name, param in net.named_parameters():
                if 'layer4' not in name and 'fc' not in name:
                    param.requires_grad = False
            print('  Backbone frozen  : all layers except layer4 + fc')
        else:
            print('  Backbone         : fully trainable (ResNet-18)')
        in_features = net.fc.in_features
        net.fc = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, 1))
    else:
        raise ValueError(f'Unknown backbone: {backbone}. Use efficientnet_b0 or resnet18.')
    return net

stage2_model = get_anomaly_classifier(
    backbone=CONFIG_S2['backbone'],
    freeze_backbone=CONFIG_S2['freeze_backbone']
).to(device)

n_pos = sum(s['label'] for s in train_s)
n_neg = len(train_s) - n_pos
pos_weight = torch.tensor([n_neg / max(1, n_pos)], dtype=torch.float32).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer_s2 = torch.optim.Adam(
    stage2_model.parameters(),
    lr=CONFIG_S2['lr'],
    weight_decay=CONFIG_S2['weight_decay']
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_s2, mode='max',
    patience=CONFIG_S2['lr_patience'],
    factor=CONFIG_S2['lr_factor'],
    verbose=True
)

total_p = sum(p.numel() for p in stage2_model.parameters())
train_p = sum(p.numel() for p in stage2_model.parameters() if p.requires_grad)
print(f'  Total params     : {total_p:,} | Trainable: {train_p:,}')
print(f'  pos_weight       : {pos_weight.item():.3f}  ({n_neg} normal / {n_pos} anomaly)')
print(f'  Loss             : BCEWithLogitsLoss | Optimizer: Adam lr={CONFIG_S2["lr"]}')
print('  ✓ Model ready')

In [ ]:
# ================== CELL 7: METRICS ==================
print('='*60)
print('STAGE 2 — CELL 7: Defining metrics')
print('='*60)

def _trapezoid_auc(fpr_list, tpr_list):
    pairs = sorted(zip(fpr_list, tpr_list))
    auc = 0.0
    for i in range(1, len(pairs)):
        dx = pairs[i][0] - pairs[i-1][0]
        auc += dx * (pairs[i][1] + pairs[i-1][1]) / 2
    return auc

def compute_auc(all_probs, all_labels):
    probs  = all_probs.numpy().ravel().astype(float)
    labels = all_labels.numpy().ravel().astype(int)
    if len(np.unique(labels)) < 2:
        return float('nan')
    if SKLEARN_OK:
        return float(roc_auc_score(labels, probs))
    thresholds = np.linspace(0, 1, 101)
    fprs, tprs = [], []
    for t in thresholds:
        preds = (probs >= t).astype(int)
        tp = int(((preds==1)&(labels==1)).sum())
        tn = int(((preds==0)&(labels==0)).sum())
        fp = int(((preds==1)&(labels==0)).sum())
        fn = int(((preds==0)&(labels==1)).sum())
        fprs.append(fp / max(1, fp+tn))
        tprs.append(tp / max(1, tp+fn))
    return _trapezoid_auc(fprs, tprs)

def compute_metrics(logits, labels, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    tp = int(((preds==1)&(labels==1)).sum())
    tn = int(((preds==0)&(labels==0)).sum())
    fp = int(((preds==1)&(labels==0)).sum())
    fn = int(((preds==0)&(labels==1)).sum())
    acc         = (tp+tn) / max(1, tp+tn+fp+fn)
    precision   = tp / max(1, tp+fp)
    recall      = tp / max(1, tp+fn)
    specificity = tn / max(1, tn+fp)
    f1          = 2*precision*recall / max(1e-8, precision+recall)
    auc         = compute_auc(probs.cpu().float(), labels.cpu().float())
    return {'acc': acc, 'precision': precision, 'recall': recall,
            'specificity': specificity, 'f1': f1, 'auc': auc,
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}

print('  Metrics: Accuracy, Precision, Recall, Specificity, F1, AUC-ROC')
print('  Best model saved by: Validation F1')
print('  ✓ Metrics defined')

In [ ]:
# ================== CELL 8: TRAINING ==================
print('='*60)
print('STAGE 2 — CELL 8: Training')
print('='*60)

best_val_f1       = -1.0
best_epoch        = 0
early_stop_counter = 0
history            = {'train': [], 'val': []}
start_time         = time.time()

print(f'  Epochs: {CONFIG_S2["epochs"]} | Batch: {CONFIG_S2["batch_size"]} | Device: {device}')
print(f'  Early stopping patience: {CONFIG_S2["early_stop_patience"]} epochs')
print(f'  Started: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('-'*70)

for epoch in range(CONFIG_S2['epochs']):
    epoch_start = time.time()

    stage2_model.train()
    t_loss, t_logits_all, t_labels_all = 0.0, [], []
    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1)
        logits = stage2_model(images)
        loss   = criterion(logits, labels)
        optimizer_s2.zero_grad()
        loss.backward()
        optimizer_s2.step()
        t_loss += loss.item() * images.size(0)
        t_logits_all.append(logits.detach().cpu())
        t_labels_all.append(labels.detach().cpu())

    t_labels_hard = (torch.cat(t_labels_all) > 0.5).float()
    tm = compute_metrics(torch.cat(t_logits_all), t_labels_hard, CONFIG_S2['binary_threshold'])
    avg_t_loss = t_loss / len(train_ds)

    stage2_model.eval()
    v_loss, v_logits_all, v_labels_all = 0.0, [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).unsqueeze(1)
            logits = stage2_model(images)
            v_loss += criterion(logits, labels).item() * images.size(0)
            v_logits_all.append(logits.cpu())
            v_labels_all.append(labels.cpu())

    vm = compute_metrics(torch.cat(v_logits_all), torch.cat(v_labels_all), CONFIG_S2['binary_threshold'])
    avg_v_loss = v_loss / len(val_ds)
    history['train'].append({'loss': avg_t_loss, **tm})
    history['val'].append({'loss': avg_v_loss, **vm})
    scheduler.step(vm['f1'])
    current_lr = optimizer_s2.param_groups[0]['lr']

    saved = ''
    if vm['f1'] > best_val_f1:
        best_val_f1        = vm['f1']
        best_epoch         = epoch + 1
        early_stop_counter = 0
        torch.save({
            'epoch': epoch+1,
            'model_state_dict': stage2_model.state_dict(),
            'optimizer_state_dict': optimizer_s2.state_dict(),
            'val_f1': best_val_f1,
            'val_auc': vm['auc'],
            'config': CONFIG_S2,
        }, CHECKPOINT_BEST)
        saved = '  ← BEST SAVED'
    else:
        early_stop_counter += 1

    auc_str = f"{vm['auc']:.4f}" if not (isinstance(vm['auc'], float) and vm['auc'] != vm['auc']) else 'N/A'
    print(
        f"Ep {epoch+1:02d}/{CONFIG_S2['epochs']} [{time.time()-epoch_start:.1f}s]"
        f" | Train loss={avg_t_loss:.4f} f1={tm['f1']:.4f} auc={tm['auc']:.4f}"
        f" | Val loss={avg_v_loss:.4f} f1={vm['f1']:.4f} "
        f"prec={vm['precision']:.4f} rec={vm['recall']:.4f} "
        f"spec={vm['specificity']:.4f} auc={auc_str}"
        f" | lr={current_lr:.2e}{saved}"
    )

    if early_stop_counter >= CONFIG_S2['early_stop_patience']:
        print(f'\n  Early stopping triggered at epoch {epoch+1} '
              f'(no F1 improvement for {CONFIG_S2["early_stop_patience"]} epochs)')
        break

print('-'*70)
print(f'  ✓ Training complete in {(time.time()-start_time)/60:.1f} min'
      f' | Best val F1: {best_val_f1:.4f} @ epoch {best_epoch}')
torch.save(stage2_model.state_dict(), CHECKPOINT_FINAL)
print(f'  ✓ Final weights saved: {CHECKPOINT_FINAL}')

In [ ]:
# ================== CELL 9: TEST EVALUATION ==================
print('='*60)
print('STAGE 2 — CELL 9: Test set evaluation + Confusion Matrix')
print('='*60)

ckpt = torch.load(CHECKPOINT_BEST, map_location=device)
stage2_model.load_state_dict(ckpt['model_state_dict'])
stage2_model.eval()
print(f"  Loaded: epoch {ckpt['epoch']}  val_f1={ckpt['val_f1']:.4f}"
      f"  val_auc={ckpt.get('val_auc', float('nan')):.4f}"
      f" | Evaluating on {len(test_ds)} samples")

test_logits, test_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        test_logits.append(stage2_model(images.to(device)).cpu())
        test_labels.append(labels.unsqueeze(1))

all_logits = torch.cat(test_logits)
all_labels = torch.cat(test_labels)
tm_test = compute_metrics(all_logits, all_labels, CONFIG_S2['binary_threshold'])

print(f"\n  ── Test Results ────────────────────────────")
print(f"  Accuracy     : {tm_test['acc']:.4f}  ({tm_test['acc']*100:.2f}%)")
print(f"  Precision    : {tm_test['precision']:.4f}")
print(f"  Recall       : {tm_test['recall']:.4f}")
print(f"  Specificity  : {tm_test['specificity']:.4f}")
print(f"  F1 Score     : {tm_test['f1']:.4f}")
print(f"  AUC-ROC      : {tm_test['auc']:.4f}")
print(f"  TP:{tm_test['tp']}  TN:{tm_test['tn']}  FP:{tm_test['fp']}  FN:{tm_test['fn']}")

cm = np.array([[tm_test['tn'], tm_test['fp']],
               [tm_test['fn'], tm_test['tp']]])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred Normal','Pred Anomaly'],
            yticklabels=['True Normal','True Anomaly'],
            linewidths=0.5, linecolor='#ccc')
ax.set_title('Confusion Matrix — Test Set', fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/stage2_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('  ✓ Confusion matrix saved: /kaggle/working/stage2_confusion_matrix.png')

report = {
    'timestamp'   : datetime.datetime.now().isoformat(),
    'best_val_f1' : best_val_f1,
    'best_epoch'  : best_epoch,
    'test_metrics': tm_test,
    'history'     : history,
    'config'      : CONFIG_S2,
}
with open(REPORT_PATH, 'w') as f:
    json.dump(report, f, indent=2)
print(f'\n  ✓ Report saved: {REPORT_PATH}')

In [ ]:
# ================== CELL 10: TRAINING CURVES ==================
print('='*60)
print('STAGE 2 — CELL 10: Plotting training curves')
print('='*60)

epochs_range = range(1, len(history['train'])+1)
fig, axes = plt.subplots(1, 3, figsize=(22, 5))
fig.suptitle('Stage 2 — Training Curves', fontsize=15, fontweight='bold')

axes[0].plot(epochs_range, [h['loss'] for h in history['train']],
             label='Train Loss', color='#01696f', linewidth=2)
axes[0].plot(epochs_range, [h['loss'] for h in history['val']],
             label='Val Loss',   color='#964219', linewidth=2)
axes[0].set_title('Loss per Epoch')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [h['f1'] for h in history['train']],
             label='Train F1', color='#01696f', linewidth=2)
axes[1].plot(epochs_range, [h['f1'] for h in history['val']],
             label='Val F1',   color='#964219', linewidth=2)
axes[1].axhline(best_val_f1, color='#aaa', linestyle='--',
                label=f'Best val F1={best_val_f1:.3f}')
axes[1].set_title('F1 per Epoch')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

train_aucs = [h['auc'] for h in history['train']]
val_aucs   = [h['auc'] for h in history['val']]
axes[2].plot(epochs_range, train_aucs,
             label='Train AUC', color='#01696f', linewidth=2)
axes[2].plot(epochs_range, val_aucs,
             label='Val AUC',   color='#964219', linewidth=2)
axes[2].set_title('AUC-ROC per Epoch')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('AUC-ROC')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/stage2_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('  ✓ Curves saved: /kaggle/working/stage2_curves.png')

In [ ]:
# ================== CELL 11: INFERENCE HELPERS (BRIDGE) ==================
print('='*60)
print('STAGE 2 — CELL 11: Inference helpers (Stage 1 → Stage 2 bridge)')
print('='*60)

_EVAL_TFM = transforms.Compose([
    transforms.Resize((CONFIG_S2['img_size'], CONFIG_S2['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def crop_tooth_from_box(image_np, box, padding=CONFIG_S2['crop_padding']):
    h, w = image_np.shape[:2]
    x1 = max(0, int(box[0]) - padding); y1 = max(0, int(box[1]) - padding)
    x2 = min(w, int(box[2]) + padding); y2 = min(h, int(box[3]) + padding)
    return Image.fromarray(image_np[y1:y2, x1:x2])

def predict_tooth(crop_pil, model, device,
                  img_size=CONFIG_S2['img_size'],
                  threshold=CONFIG_S2['binary_threshold']):
    x = _EVAL_TFM(crop_pil).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(x)).item()
    return {
        'label': 'ANOMALY' if prob >= threshold else 'NORMAL',
        'probability_anomaly': round(prob, 4)
    }

def predict_teeth_batch(crops_pil, model, device,
                        img_size=CONFIG_S2['img_size'],
                        threshold=CONFIG_S2['binary_threshold'],
                        batch_size=16):
    model.eval()
    results = []
    for i in range(0, len(crops_pil), batch_size):
        batch_crops = crops_pil[i:i+batch_size]
        tensors = torch.stack([_EVAL_TFM(c) for c in batch_crops]).to(device)
        with torch.no_grad():
            probs = torch.sigmoid(model(tensors)).cpu().squeeze(1).tolist()
        for p in probs:
            results.append({
                'label': 'ANOMALY' if p >= threshold else 'NORMAL',
                'probability_anomaly': round(float(p), 4)
            })
    return results

def run_stage2_on_stage1_output(image_np, stage1_predictions, stage2_model, device,
                                threshold=CONFIG_S2['binary_threshold'],
                                use_batch=True):
    boxes  = stage1_predictions['boxes']
    labels = stage1_predictions['labels']
    scores = stage1_predictions['scores']
    crops  = [crop_tooth_from_box(image_np, b) for b in boxes]
    if use_batch and len(crops) > 0:
        preds = predict_teeth_batch(crops, stage2_model, device, threshold=threshold)
    else:
        preds = [predict_tooth(c, stage2_model, device, threshold=threshold) for c in crops]
    return [
        {
            'tooth_id'           : int(tid),
            'box'                : b.tolist() if hasattr(b, 'tolist') else list(b),
            'segmentation_score' : round(float(sc), 4),
            'anomaly_label'      : p['label'],
            'anomaly_probability': p['probability_anomaly'],
        }
        for tid, b, sc, p in zip(labels, boxes, scores, preds)
    ]

print('  crop_tooth_from_box()            — crops one tooth from Stage 1 box')
print('  predict_tooth()                  — single-crop inference')
print('  predict_teeth_batch()            — batch inference (faster for full images)')
print('  run_stage2_on_stage1_output()    — full Stage 1 → Stage 2 bridge')
print('  ✓ Bridge functions defined')

In [ ]:
# ================== CELL 12: COMBINED VISUALIZATION ==================
print('='*60)
print('STAGE 2 — CELL 12: Defining combined pipeline visualization')
print('='*60)

def visualize_pipeline(image_tensor, stage1_model, stage2_model, device,
                       sample_index=0, s1_conf=0.6,
                       s2_thresh=CONFIG_S2['binary_threshold']):
    print(f'\n  Processing sample index {sample_index}...')
    image_np = (image_tensor.permute(1,2,0).numpy()*255).astype(np.uint8).copy()

    stage1_model.eval()
    with torch.no_grad():
        from torchvision.ops import nms as _nms
        raw  = stage1_model(image_tensor.to(device).unsqueeze(0))[0]
        keep = raw['scores'] >= s1_conf
        filt = {k: v[keep] for k, v in raw.items()}
        if len(filt['boxes']) > 0:
            ki   = _nms(filt['boxes'], filt['scores'], 0.3)
            filt = {k: v[ki] for k, v in filt.items()}
        stage1_preds = {
            'boxes' : filt['boxes'].cpu().numpy(),
            'labels': filt['labels'].cpu().numpy(),
            'masks' : filt['masks'].cpu().numpy(),
            'scores': filt['scores'].cpu().numpy(),
        }
    print(f'    Stage 1: {len(stage1_preds["labels"])} teeth detected')

    results = run_stage2_on_stage1_output(
        image_np, stage1_preds, stage2_model, device,
        threshold=s2_thresh, use_batch=True
    )
    n_anom = sum(1 for r in results if r['anomaly_label'] == 'ANOMALY')
    print(f'    Stage 2: {n_anom}/{len(results)} teeth flagged as anomaly')

    overlay = image_np.copy()
    NORMAL_COLOR  = (40,  200,  80)
    ANOMALY_COLOR = (220, 40,   40)
    for r, mask_raw in zip(results, stage1_preds['masks']):
        mask_bin = (mask_raw[0] > 0.5).astype(np.uint8)
        color    = ANOMALY_COLOR if r['anomaly_label'] == 'ANOMALY' else NORMAL_COLOR
        for c in range(3):
            ch = overlay[:,:,c].copy().astype(np.float32)
            ch[mask_bin==1] = 0.35*color[c] + 0.65*ch[mask_bin==1]
            overlay[:,:,c] = ch.astype(np.uint8)
        contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, color, 3)
        cx = int((r['box'][0]+r['box'][2])//2)
        cy = int((r['box'][1]+r['box'][3])//2)
        cv2.putText(overlay, f"T{r['tooth_id']}",
                    (cx-20, cy+8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0),     4)
        cv2.putText(overlay, f"T{r['tooth_id']}",
                    (cx-20, cy+8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        if r['anomaly_label'] == 'ANOMALY':
            cv2.putText(overlay, f"{r['anomaly_probability']:.2f}",
                        (cx-18, cy+28), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,220,0), 2)

    fig, axes = plt.subplots(1, 2, figsize=(24, 10))
    fig.suptitle(
        f"Pipeline Sample {sample_index} — {n_anom}/{len(results)} anomalies",
        fontsize=15, fontweight='bold'
    )
    axes[0].imshow(image_np, cmap='gray')
    axes[0].set_title('Original Panoramic X-ray', fontsize=13)
    axes[0].axis('off')
    axes[1].imshow(overlay)
    axes[1].set_title('Segmented + Anomaly Classified', fontsize=13)
    axes[1].axis('off')
    axes[1].legend(
        handles=[
            mpatches.Patch(color=(40/255,200/255,80/255),  label='Normal'),
            mpatches.Patch(color=(220/255,40/255,40/255),  label='Anomaly'),
        ],
        loc='lower right', fontsize=12
    )
    plt.tight_layout()
    out_path = f'/kaggle/working/pipeline_sample_{sample_index}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'    ✓ Saved: {out_path}')

    print(f"\n    ── Tooth Report — Sample {sample_index} ──")
    print(f"    {'Tooth':>6}  {'Label':<12}  {'P(anomaly)':>12}  {'SegScore':>10}")
    print(f"    {'─'*45}")
    for r in results:
        flag = '🔴 ANOMALY' if r['anomaly_label'] == 'ANOMALY' else '🟢 Normal '
        print(f"    T{r['tooth_id']:>2d}    {flag}    {r['anomaly_probability']:>10.4f}    {r['segmentation_score']:>8.4f}")
    print(f"\n    Summary: {n_anom}/{len(results)} anomaly | {len(results)-n_anom}/{len(results)} normal")

print('  visualize_pipeline() defined')
print('  ✓ Visualization helper ready')

In [ ]:
# ================== CELL 13: RUN FULL PIPELINE ON TEST SAMPLES ==================
print('='*60)
print('STAGE 2 — CELL 13: Running full pipeline on test samples')
print('='*60)

# NOTE: `model`        = Stage 1 Mask R-CNN (from stage1 notebook, still in memory)
# NOTE: `test_dataset` = Stage 1 test split  (from stage1 notebook, still in memory)

NUM_SAMPLES = min(3, len(test_dataset))
print(f'  Running on {NUM_SAMPLES} test samples from Stage 1 test set')

for i in range(NUM_SAMPLES):
    print(f"\n{'='*60}\n  SAMPLE {i+1} / {NUM_SAMPLES}\n{'='*60}")
    image_tensor, _ = test_dataset[i]
    visualize_pipeline(
        image_tensor=image_tensor,
        stage1_model=model,
        stage2_model=stage2_model,
        device=device,
        sample_index=i+1
    )

print(f"\n{'='*60}")
print('  ✓ STAGE 2 COMPLETE')
print(f'  Checkpoints : {CHECKPOINT_BEST}')
print(f'               {CHECKPOINT_FINAL}')
print(f'  Report      : {REPORT_PATH}')
print(f'  Curves      : /kaggle/working/stage2_curves.png')
print(f'  Confusion M : /kaggle/working/stage2_confusion_matrix.png')
print(f'  Visuals     : /kaggle/working/pipeline_sample_*.png')
print(f"{'='*60}")